# 06.13 - Deep Learning Synthesis & Review

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

A cumulative integration unit combining all Phase 06 concepts — perceptrons, losses, backprop, PyTorch, regularization, optimizers, data loading, training loops, CNNs, RNNs, attention — into an end-to-end pipeline.

## 2. Why Does This Matter?

Knowing individual concepts isn't enough. You must build an end-to-end deep learning system and make architectural and training decisions independently.

## 3. Prerequisites

- All prior Phase 06 units

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Assemble a full PyTorch training pipeline
- Apply regularization, scheduling, checkpointing, early stopping
- Compare configurations and evaluate fairly
- Debug common training failures

## 5. Mental Model

Everything connects: data → model → train/validate → regularize → schedule → checkpoint, with monitoring at every step.

> Project uses synthetic image-like spatial data and a custom CNN (no downloads).


## 6. Backend + Synthetic Spatial Dataset


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
from sklearn.metrics import confusion_matrix

torch.manual_seed(42); np.random.seed(42)

def synth_images(n, seed=0):
    g = np.random.default_rng(seed)
    X = g.standard_normal((n, 1, 8, 8)) * 0.1
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        if i % 3 == 0:
            X[i, 0, 1:4, 1:4] += 1.0; y[i] = 0      # top-left
        elif i % 3 == 1:
            X[i, 0, 4:7, 4:7] += 1.0; y[i] = 1      # bottom-right
        else:
            X[i, 0, 3:5, 3:5] += 1.0; y[i] = 2      # center
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

X, y = synth_images(1200)
n = len(X)
tr, val, te = int(0.6*n), int(0.2*n), n - int(0.8*n)
Xtr, ytr = X[:tr], y[:tr]
Xval, yval = X[tr:tr+val], y[tr:tr+val]
Xte, yte = X[tr+val:], y[tr+val:]
print(f"Split sizes: train={tr}, val={val}, test={te}")
print("Classes:", sorted(y.unique().tolist()))


Split sizes: train=720, val=240, test=240


Classes: [0, 1, 2]


## 7. Custom CNN + Regularization + Optimizer + Scheduler

Assemble all the pieces with dropout, weight decay, and cosine annealing.


In [2]:
class SynthCNN(nn.Module):
    def __init__(self, num_classes=3, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*2*2, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = SynthCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print("CNN + BatchNorm + Dropout + AdamW + cosine schedule assembled.")


Params: 3,475
CNN + BatchNorm + Dropout + AdamW + cosine schedule assembled.


## 8. Complete Train / Validate Loop with Checkpointing

Track train and validation loss/accuracy; keep the best state.


In [3]:
def accuracy(pred, tgt):
    return (pred.argmax(dim=1) == tgt).float().mean().item()

@torch.no_grad()
def evaluate(model, Xv, yv):
    model.eval()
    pred = model(Xv)
    loss = criterion(pred, yv).item()
    return loss, accuracy(pred, yv), pred.argmax(dim=1)

history = {"tr_loss": [], "va_loss": [], "va_acc": []}
best_val = float('inf'); best_state = None
os.makedirs('_tmp_syn', exist_ok=True)
for epoch in range(30):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(Xtr), ytr)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    vl, va, _ = evaluate(model, Xval, yval)
    history["tr_loss"].append(loss.item())
    history["va_loss"].append(vl)
    history["va_acc"].append(va)
    if vl < best_val:
        best_val = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:2d}: tr_loss={loss.item():.4f} va_loss={vl:.4f} va_acc={va:.3f}")

model.load_state_dict(best_state)
print(f"\nBest val loss {best_val:.4f} (model restored to best checkpoint).")


epoch 10: tr_loss=0.0748 va_loss=0.5750 va_acc=1.000


epoch 20: tr_loss=0.0095 va_loss=0.1054 va_acc=1.000


epoch 30: tr_loss=0.0110 va_loss=0.0202 va_acc=1.000

Best val loss 0.0202 (model restored to best checkpoint).


## 9. Evaluate: Accuracy, Per-Class, Confusion Matrix


In [4]:
te_loss, te_acc, te_pred = evaluate(model, Xte, yte)
cm = confusion_matrix(yte.numpy(), te_pred.numpy(), labels=[0, 1, 2])
print(f"Test accuracy: {te_acc:.3f}")
print("Confusion matrix (rows=truth, cols=pred):")
print(cm)
per_class = cm.diagonal() / cm.sum(axis=1).clip(min=1)
print("Per-class accuracy:", np.round(per_class, 3))
print("\nConfusion matrix flags which classes get confused — here it's clean.")


Test accuracy: 1.000
Confusion matrix (rows=truth, cols=pred):
[[80  0  0]
 [ 0 80  0]
 [ 0  0 80]]
Per-class accuracy: [1. 1. 1.]

Confusion matrix flags which classes get confused — here it's clean.


## 10. Ablation: With vs Without Regularization

Fairly compare two configurations.


In [5]:
def train_cfg(use_reg, epochs=25):
    m = SynthCNN(dropout=0.3 if use_reg else 0.0)
    crit = nn.CrossEntropyLoss()
    opt = optim.AdamW(m.parameters(), lr=0.005, weight_decay=1e-3 if use_reg else 0.0)
    for _ in range(epochs):
        m.train(); opt.zero_grad(); crit(m(Xtr), ytr).backward(); opt.step()
    _, acc, _ = evaluate(m, Xte, yte)
    return acc

reg_acc   = train_cfg(True)
noreg_acc = train_cfg(False)
print(f"With  regularization (dropout+wd): test acc = {reg_acc:.3f}")
print(f"Without regularization:            test acc = {noreg_acc:.3f}")
print("\nRegularization helps generalization here (or at least doesn't hurt).")


With  regularization (dropout+wd): test acc = 1.000
Without regularization:            test acc = 1.000

Regularization helps generalization here (or at least doesn't hurt).


## 11. Review: Required Decision Comparisons

| Comparison | Use First | Use Second | Trade-off |
|---|---|---|---|
| MLP vs CNN | Tabular | Image/spatial | Simplicity vs spatial |
| RNN vs Transformer | Short seq | Long seq | Speed vs expressiveness |
| Dropout vs BN | Simple models | Deep nets | Different effects |
| Adam vs SGD | Prototyping | Final | Speed vs generalization gap |
| Feat-extract vs fine-tune | Small/similar | Large/domain shift | Data efficiency vs max perf |
| From scratch vs pretrained | Learning | Limited data | Understanding vs practicality |

## 12. Common Failure Cases

| Symptom | Cause | Fix |
|---|---|---|
| Overfits immediately | Too many params | Dropout, smaller model, augmentation |
| NaN loss | High LR / bad data | Lower LR, clipping, check data |
| Accuracy stuck | Underfit / wrong loss | More capacity, verify loss |
| Transfer worse | Features destroyed | Freeze, lower LR |
| RNN no long patterns | Vanishing gradients | LSTM/GRU |

## 13. Phase Mastery Check

Without a tutorial, you should be able to:

1. Implement backprop from scratch in NumPy
2. Build/train MLPs, CNNs, RNNs in PyTorch
3. Choose correct loss functions
4. Apply regularization
5. Select/configure optimizers and schedules
6. Build efficient data pipelines
7. Implement full training loops with validation
8. Save/load checkpoints
9. Use transfer learning
10. Implement attention from scratch
11. Debug NaN/overfitting/vanishing gradients
12. Build an end-to-end system independently

## 14. Challenge (Extended)

Extend the model with a lightweight attention readout over the flattened CNN features, then compare to the plain classifier.


In [6]:
# Challenge: attention-augmented readout vs plain head
class AttnHead(nn.Module):
    def __init__(self, d=32, ncls=3):
        super().__init__()
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d)
        self.fc = nn.Linear(d, ncls)
    def forward(self, z):  # z: (B, T, d)
        w = torch.softmax((self.q(z) @ self.k(z).transpose(-2,-1)) / (z.size(-1)**0.5), dim=-1)
        ctx = (w @ self.v(z)).mean(dim=1)
        return self.fc(ctx)

class SynthCNN_Attn(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.proj = nn.Linear(16*2*2, 32)
        self.head = AttnHead(32)
    def forward(self, x):
        f = self.features(x).flatten(1)              # (B, 64)
        z = self.proj(f).unsqueeze(1)                # (B, 1, 32) - single token
        return self.head(z)

m_attn = SynthCNN_Attn(); crit = nn.CrossEntropyLoss()
opt = optim.Adam(m_attn.parameters(), lr=0.005)
for __ in range(25):
    m_attn.train(); opt.zero_grad(); crit(m_attn(Xtr), ytr).backward(); opt.step()
_, acc_attn, _ = evaluate(m_attn, Xte, yte)
print(f"Attention-augmented CNN test acc: {acc_attn:.3f}  (baseline CNN was about {te_acc:.3f})")
print("\nAttention readout is a bridge to transformers (Phase 07).")


Attention-augmented CNN test acc: 1.000  (baseline CNN was about 1.000)

Attention readout is a bridge to transformers (Phase 07).


## 15. Closed-Book Recall

Without looking back:

1. Why did you choose CNN over MLP for images?
2. Which regularization did you use and why?
3. How did you verify the model isn't overfitting?
4. What would you change with more data/time?

## 16. Teach-Back Questions

Explain to another person:

- The full end-to-end pipeline and every decision.
- How to debug a model with low train loss but high val loss.
- When to use CNN vs RNN vs MLP.

## 17. Summary

You built an end-to-end CNN pipeline with regularization, scheduling, checkpointing, validation, evaluation, and ablations — integrating every Phase 06 concept.

## 18. Further Experiment

- Add data augmentation to the pipeline.
- Train an RNN on the same raw features via a flattened sequence view.

## 19. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
